# 난임 환자 임신 성공 여부 예측 

## 구성
1. 라이브러리 & 데이터 로드
2. 결측치 / 이상치 처리
3. 피처 엔지니어링
4. 피처 정제 + 교호작용 추가
5. 피처 준비 (3모델용)
6. Seed Averaging 3모델 앙상블
7. 제출 파일 생성

## 최종 성능
- OOF AUC: **0.74069** (Seed Averaging 5 SEED)
- 리더보드: **0.74193** (3모델 Optuna 튜닝 앙상블 기준)


---
# Section 0. 라이브러리 & 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OrdinalEncoder
import warnings
warnings.filterwarnings('ignore')

# ── 경로 설정 ─────────────────────────────────────────────────────
DATA_DIR = '/Users/Jiyeon/Desktop/ftp/fertility-treatment-prediction/data/raw'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')

TARGET = '임신 성공 여부'
ID_COL = 'ID'

print('Train:', train.shape, '/ Test:', test.shape)
print('타겟 분포:')
print(train[TARGET].value_counts(normalize=True).round(3))

---
# Section 1. 결측치 / 이상치 처리

## 핵심: 구조적 결측(Structural Missingness)
- DI(인공수정)는 배아 생성·이식 과정 자체가 없음 → 결측 = 실제 0
- 일반 결측치처럼 평균/중앙값으로 채우면 모델이 시술 차이를 왜곡 학습
- **처리 순서 중요**: 플래그 생성 → 0 대체 (반드시 이 순서)

In [ ]:
# ── 횟수 컬럼 정수 변환 ───────────────────────────────────────────
COUNT_MAP = {'0회':0,'1회':1,'2회':2,'3회':3,'4회':4,'5회':5,'6회 이상':6}
COUNT_COLS = [
    '총 시술 횟수','클리닉 내 총 시술 횟수','IVF 시술 횟수','DI 시술 횟수',
    '총 임신 횟수','IVF 임신 횟수','DI 임신 횟수',
    '총 출산 횟수','IVF 출산 횟수','DI 출산 횟수',
]
for df in [train, test]:
    for col in COUNT_COLS:
        if col in df.columns and df[col].dtype == 'object':
            df[col] = df[col].map(COUNT_MAP)

is_di_train = train['시술 유형'] == 'DI'
is_di_test  = test['시술 유형']  == 'DI'
print('횟수 변환 완료')

In [ ]:
# ── STEP 1: 플래그 생성 (반드시 0 대체 전에) ──────────────────────
FLAG_MAP = {
    '난자채취_진입여부': '난자 채취 경과일',
    '배아이식_진입여부': '배아 이식 경과일',
    '배아해동_진입여부': '배아 해동 경과일',
    '난자혼합_진입여부': '난자 혼합 경과일',
    'ICSI시술_진입여부': '미세주입된 난자 수',
    '배아저장_진입여부': '저장된 배아 수',
}
for flag, src in FLAG_MAP.items():
    if src in train.columns:
        train[flag] = train[src].notna().astype(int)
        test[flag]  = test[src].notna().astype(int)

# 시술 중단 플래그
for df, is_di in [(train, is_di_train), (test, is_di_test)]:
    is_ivf = ~is_di
    egg_collected  = df['수집된 신선 난자 수'].fillna(0) > 0
    no_embryo_made = df['총 생성 배아 수'].fillna(0) == 0
    no_transfer    = df['이식된 배아 수'].fillna(0) == 0
    embryo_made    = df['총 생성 배아 수'].fillna(0) > 0

    df['배아생성_실패_플래그'] = (is_ivf & egg_collected & no_embryo_made).astype(int)
    df['이식취소_플래그']      = (is_ivf & embryo_made & no_transfer).astype(int)
    df['사이클중단_플래그']    = (is_ivf & ~egg_collected & df['수집된 신선 난자 수'].notna()).astype(int)

print('플래그 생성 완료')

In [ ]:
# ── STEP 2: DI 배아 개수 → 0 대체 ────────────────────────────────
ZERO_IMPUTE_COLS = [
    '총 생성 배아 수','미세주입된 난자 수','미세주입에서 생성된 배아 수',
    '이식된 배아 수','미세주입 배아 이식 수','저장된 배아 수',
    '미세주입 후 저장된 배아 수','해동된 배아 수','해동 난자 수',
    '수집된 신선 난자 수','저장된 신선 난자 수','혼합된 난자 수',
    '파트너 정자와 혼합된 난자 수','기증자 정자와 혼합된 난자 수',
]
ZERO_IMPUTE_COLS = [c for c in ZERO_IMPUTE_COLS if c in train.columns]
for col in ZERO_IMPUTE_COLS:
    train.loc[is_di_train & train[col].isnull(), col] = 0
    test.loc[is_di_test   & test[col].isnull(),  col] = 0

# ── STEP 3: DI 경과일 → -1 특이값 ────────────────────────────────
ELAPSED_COLS = [
    '난자 채취 경과일','난자 해동 경과일','난자 혼합 경과일',
    '배아 이식 경과일','배아 해동 경과일'
]
ELAPSED_COLS = [c for c in ELAPSED_COLS if c in train.columns]
for col in ELAPSED_COLS:
    train.loc[is_di_train & train[col].isnull(), col] = -1
    test.loc[is_di_test   & test[col].isnull(),  col] = -1

# ── STEP 4: 범주형 결측 → 'missing' ──────────────────────────────
cat_cols_raw = [c for c in train.columns if train[c].dtype == 'object']
for df in [train, test]:
    for col in cat_cols_raw:
        df[col] = df[col].fillna('missing')

# ── STEP 5: 고결측 변수 플래그 ────────────────────────────────────
HIGH_MISS = [
    '임신 시도 또는 마지막 임신 경과 연수',
    '착상 전 유전 검사 사용 여부','착상 전 유전 진단 사용 여부',
    'PGS 시술 여부','PGD 시술 여부','난자 해동 경과일',
]
for col in [c for c in HIGH_MISS if c in train.columns]:
    for df in [train, test]:
        df[f'{col}_결측여부'] = df[col].isnull().astype(int)

# ── STEP 6: 이상치 처리 ───────────────────────────────────────────
# 음수 개수값 → NaN
for df in [train, test]:
    for col in ZERO_IMPUTE_COLS:
        df.loc[df[col] < 0, col] = np.nan

# 도메인 상한 클리핑
DOMAIN_UPPER = {
    '수집된 신선 난자 수': 50, '혼합된 난자 수': 50,
    '총 생성 배아 수': 40, '이식된 배아 수': 5,
    '저장된 배아 수': 30,
}
for df in [train, test]:
    for col, upper in DOMAIN_UPPER.items():
        if col in df.columns:
            df[col] = df[col].clip(upper=upper)

print('결측치/이상치 처리 완료')

---
# Section 2. 피처 엔지니어링

난임 임상 도메인 지식 기반 120개 파생 피처 생성

In [ ]:
def safe_ratio(num, denom):
    return np.where(denom > 0, num / denom, 0)

AGE_MAP = {
    '만18-34세':1,'만35-37세':2,'만38-39세':3,
    '만40-42세':4,'만43-44세':5,'만45-50세':6,'알 수 없음':np.nan,'missing':np.nan
}
AGE_PRIOR = {
    '만18-34세':0.32,'만35-37세':0.27,'만38-39세':0.22,
    '만40-42세':0.16,'만43-44세':0.12,'만45-50세':0.17,'알 수 없음':np.nan,'missing':np.nan
}
DONOR_AGE_MAP = {
    '만20세 이하':1,'만21-25세':2,'만26-30세':3,
    '만31-35세':4,'만36-40세':5,'만41-45세':6,'알 수 없음':np.nan,'missing':np.nan
}

for df in [train, test]:
    is_di = df['시술 유형'] == 'DI'

    # ── 나이 피처 ─────────────────────────────────────────────────
    df['나이_순서']            = df['시술 당시 나이'].map(AGE_MAP)
    df['나이_성공률_사전점수'] = df['시술 당시 나이'].map(AGE_PRIOR)
    df['나이_35이상'] = (df['나이_순서'] >= 2).astype(int)
    df['나이_38이상'] = (df['나이_순서'] >= 3).astype(int)
    df['나이_40이상'] = (df['나이_순서'] >= 4).astype(int)
    df['나이_43이상'] = (df['나이_순서'] >= 5).astype(int)

    # ── 난소 예비능 (POSEIDON 기준) ───────────────────────────────
    egg_n = df['수집된 신선 난자 수'].fillna(0)
    df['난소반응_그룹']  = pd.cut(egg_n, bins=[-1,0,3,14,100],
                                   labels=['채취없음','Poor(1-3개)','Normal(4-14개)','High(15개이상)']).astype(str)
    df['poor_responder'] = ((egg_n > 0) & (egg_n < 4)).astype(int)
    df['high_responder'] = (egg_n >= 15).astype(int)
    df['난자수_log']     = np.log1p(egg_n)
    fresh_tx = df['신선 배아 사용 여부'].fillna(0).astype(int)
    stored   = df['저장된 배아 수'].fillna(0)
    df['freeze_all_전략']  = ((fresh_tx == 0) & (stored > 0)).astype(int)

    # ── 배아 품질 프록시 ──────────────────────────────────────────
    df['수정_효율']        = safe_ratio(df['총 생성 배아 수'], df['혼합된 난자 수'])
    df['ICSI_수정_효율']   = safe_ratio(df['미세주입에서 생성된 배아 수'], df['미세주입된 난자 수'])
    df['이식_효율']        = safe_ratio(df['이식된 배아 수'], df['총 생성 배아 수'])
    df['저장_비율']        = safe_ratio(df['저장된 배아 수'], df['총 생성 배아 수'])
    df['난자_배아_전환율'] = safe_ratio(df['총 생성 배아 수'], df['수집된 신선 난자 수'])
    df['파트너정자_비율']  = safe_ratio(df['파트너 정자와 혼합된 난자 수'], df['혼합된 난자 수'])
    df['배아품질_종합점수'] = (df['수정_효율']*0.4 + df['이식_효율']*0.4 + df['저장_비율']*0.2).clip(0,1)

    for col in ['총 생성 배아 수','이식된 배아 수','수집된 신선 난자 수','혼합된 난자 수','저장된 배아 수']:
        if col in df.columns:
            df[f'{col}_log'] = np.log1p(df[col].fillna(0))

    for flag, src in [('배아생성_있음','총 생성 배아 수'),('이식배아_있음','이식된 배아 수'),
                       ('배아저장_있음','저장된 배아 수'),('ICSI시술_있음','미세주입된 난자 수'),
                       ('해동배아_있음','해동된 배아 수')]:
        if src in df.columns:
            df[flag] = (df[src].fillna(0) > 0).astype(int)

    df['이식배아_구간'] = pd.cut(df['이식된 배아 수'].fillna(0),
                                  bins=[-1,0,1,2,100], labels=['0개','1개','2개','3개이상']).astype(str)
    df['잔여배아_수'] = (df['총 생성 배아 수'].fillna(0) - df['이식된 배아 수'].fillna(0)).clip(lower=0)

    # ── 시술 유형 파싱 ────────────────────────────────────────────
    proc = df['특정 시술 유형'].fillna('').str.upper()
    df['배반포이식_여부'] = proc.str.contains('BLASTOCYST').astype(int)
    df['ICSI_여부']       = proc.str.contains('ICSI').astype(int)
    df['FER_여부']        = proc.str.contains('FER').astype(int)
    df['AH_여부']         = proc.str.contains(r'\bAH\b').astype(int)
    df['IUI_여부']        = proc.str.contains('IUI').astype(int)
    df['ICSI_배반포_조합'] = ((df['ICSI_여부']==1) & (df['배반포이식_여부']==1)).astype(int)
    tech = ['ICSI_여부','배반포이식_여부','AH_여부',
            '착상 전 유전 검사 사용 여부','착상 전 유전 진단 사용 여부']
    df['기술집약도_점수'] = df[[c for c in tech if c in df.columns]].fillna(0).sum(axis=1)

    # ── Rare category 병합 ────────────────────────────────────────
    if '특정 시술 유형' in df.columns:
        proc_freq = train['특정 시술 유형'].value_counts()
        rare_proc = proc_freq[proc_freq < 50].index
        df['특정시술_정제'] = df['특정 시술 유형'].apply(
            lambda x: 'OTHER' if x in rare_proc else (x if pd.notna(x) else 'missing'))

    stim_freq = train['배란 유도 유형'].value_counts()
    rare_stim = stim_freq[stim_freq < 50].index
    df['배란유도_정제'] = df['배란 유도 유형'].apply(
        lambda x: 'OTHER' if x in rare_stim else (x if pd.notna(x) else 'missing'))

    # ── 경과일 간격 ───────────────────────────────────────────────
    df['배양_기간_일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['채취_이식_총기간'] = df['배아 이식 경과일'] - df['난자 채취 경과일']
    df['채취_혼합_간격']  = df['난자 혼합 경과일'] - df['난자 채취 경과일']
    df['해동_이식_간격']  = df['배아 이식 경과일'] - df['배아 해동 경과일']
    for col in ['배양_기간_일','채취_이식_총기간','채취_혼합_간격','해동_이식_간격']:
        df.loc[is_di, col] = df.loc[is_di, col].fillna(-1)
        df.loc[~is_di & (df[col] < 0), col] = np.nan
    df['배양기간_D5추정'] = ((df['배양_기간_일'] >= 4) & (df['배양_기간_일'] != -1)).astype(int)

    def classify_day(row):
        if row['시술 유형'] == 'DI': return 'DI_NA'
        d = row['배양_기간_일']
        if pd.isna(d) or d < 0: return '미상'
        if d <= 3.5: return 'D3_세포기'
        elif d <= 4.5: return 'D4_모룰라'
        else: return 'D5이상_배반포'
    df['이식_단계_범주'] = df.apply(classify_day, axis=1)

    # ── 시술 이력 ─────────────────────────────────────────────────
    df['총_실패_횟수']    = (df['총 시술 횟수'] - df['총 임신 횟수']).clip(lower=0)
    df['IVF_실패_횟수']   = (df['IVF 시술 횟수'] - df['IVF 임신 횟수']).clip(lower=0)
    df['RIF_플래그']      = ((df['총 시술 횟수'] >= 3) & (df['총 임신 횟수'] == 0)).astype(int)
    df['첫시도_여부']     = (df['총 시술 횟수'] == 0).astype(int)
    df['출산경험_있음']   = (df['총 출산 횟수'] > 0).astype(int)
    df['IVF출산경험_있음'] = (df['IVF 출산 횟수'] > 0).astype(int)
    df['과거_임신성공률'] = np.where(df['총 시술 횟수'] > 0,
                                      df['총 임신 횟수'] / df['총 시술 횟수'], np.nan)
    df['임신_출산전환율'] = np.where(df['총 임신 횟수'] > 0,
                                      df['총 출산 횟수'] / df['총 임신 횟수'], np.nan)
    df['클리닉_시술비율'] = np.where(df['총 시술 횟수'] > 0,
                                      df['클리닉 내 총 시술 횟수'] / df['총 시술 횟수'], np.nan)
    df['RIF_AH_조합']     = ((df['RIF_플래그']==1) & (df['AH_여부']==1)).astype(int)

    # ── 불임 원인 ─────────────────────────────────────────────────
    cause_cols = [c for c in [
        '불임 원인 - 난관 질환','불임 원인 - 남성 요인','불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인','불임 원인 - 자궁경부 문제','불임 원인 - 자궁내막증',
        '불임 원인 - 정자 농도','불임 원인 - 정자 면역학적 요인',
        '불임 원인 - 정자 운동성','불임 원인 - 정자 형태'
    ] if c in df.columns]
    sperm_cols  = [c for c in cause_cols if '정자' in c]
    female_cols = [c for c in cause_cols if c not in sperm_cols and '남성' not in c]
    df['불임원인_수']         = df[cause_cols].fillna(0).sum(axis=1)
    df['정자_문제_수']        = df[sperm_cols].fillna(0).sum(axis=1)
    df['여성_원인_수']        = df[female_cols].fillna(0).sum(axis=1)
    df['중증남성불임_플래그'] = (df['정자_문제_수'] >= 3).astype(int)
    df['복합원인_플래그']     = (df['불임원인_수'] >= 2).astype(int)
    df['원인불명_플래그']     = df['불명확 불임 원인'].fillna(0).astype(int)
    has_male   = (df['남성 주 불임 원인'].fillna(0)==1) | (df['정자_문제_수']>0)
    has_female = (df['여성 주 불임 원인'].fillna(0)==1) | (df['여성_원인_수']>0)
    has_both   = df['부부 주 불임 원인'].fillna(0)==1
    df['불임원인_주체'] = np.select(
        [has_both, has_male & ~has_female, has_female & ~has_male,
         has_male & has_female, df['원인불명_플래그'].astype(bool)],
        ['부부복합','남성단독','여성단독','남녀복합','불명확'], default='기타')

    # ── PGT ───────────────────────────────────────────────────────
    pgs = df.get('착상 전 유전 검사 사용 여부', pd.Series(0, index=df.index)).fillna(0)
    pgd = df.get('착상 전 유전 진단 사용 여부',  pd.Series(0, index=df.index)).fillna(0)
    df['PGT_사용_여부']     = ((pgs > 0) | (pgd > 0)).astype(int)
    df['고령_PGT_조합']     = ((df['나이_38이상']==1) & (df['PGT_사용_여부']==1)).astype(int)
    df['RIF_PGT_조합']      = ((df['RIF_플래그']==1) & (df['PGT_사용_여부']==1)).astype(int)

    # ── 출처 ──────────────────────────────────────────────────────
    df['난자기증자_나이_순서'] = df['난자 기증자 나이'].map(DONOR_AGE_MAP)
    df['기증난자_사용']        = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증정자_사용']        = (df['정자 출처'] == '기증 제공').astype(int)
    df['자가난자_고령_조합']   = ((df['난자 출처']=='본인 제공') & (df['나이_40이상']==1)).astype(int)
    df['자가난자_한계연령']    = ((df['난자 출처']=='본인 제공') & (df['나이_43이상']==1)).astype(int)
    df['젊은기증난자_조합']    = ((df['기증난자_사용']==1) & (df['난자기증자_나이_순서'].fillna(99)<=3)).astype(int)

    # ── 배아 생성 목적 ────────────────────────────────────────────
    reason = df['배아 생성 주요 이유'].fillna('').str
    df['현재시술용_포함'] = reason.contains('현재 시술용').astype(int)
    df['배아저장용_포함'] = reason.contains('배아 저장용').astype(int)
    df['난자저장용_포함'] = reason.contains('난자 저장용').astype(int)
    df['기증용_포함']     = reason.contains('기증용').astype(int)
    df['목적_수']         = df['현재시술용_포함']+df['배아저장용_포함']+df['난자저장용_포함']+df['기증용_포함']
    df['즉시임신단독_여부'] = ((df['현재시술용_포함']==1)&(df['배아저장용_포함']==0)&(df['난자저장용_포함']==0)).astype(int)

    # ── 상호작용 ──────────────────────────────────────────────────
    is_ivf_flag = (df['시술 유형'] == 'IVF').astype(int)
    age    = df['나이_순서'].fillna(0)
    quality = df['배아품질_종합점수'].fillna(0)
    emb_n  = df['이식된 배아 수'].fillna(0)
    trial  = df.get('임신 시도 또는 마지막 임신 경과 연수', pd.Series(0,index=df.index)).fillna(0)

    df['나이_IVF_상호작용']       = age * is_ivf_flag
    df['나이_배반포_상호작용']     = age * df['배반포이식_여부']
    df['나이_배아품질_상호작용']   = age * quality
    df['나이_시술횟수_상호작용']   = age * df['총 시술 횟수'].fillna(0)
    df['poor_resp_수정효율']       = df['poor_responder'] * df['수정_효율']
    df['고위험_3중_조합']          = ((df['나이_40이상']==1)&(df['기증난자_사용']==0)&(is_ivf_flag==1)).astype(int)
    df['난임_난이도_점수']         = age*0.4 + df['총_실패_횟수'].fillna(0)*0.3 + trial*0.15 + df['복합원인_플래그'].fillna(0)*0.15
    df['성공잠재력_점수']          = quality*0.4 + df['과거_임신성공률'].fillna(0)*0.3 + df['기술집약도_점수']/5*0.2 + df['출산경험_있음']*0.1
    df['출산경험_나이_상호작용']   = df['출산경험_있음'] * age

    # ── 장기 난임 ─────────────────────────────────────────────────
    COL_YEARS = '임신 시도 또는 마지막 임신 경과 연수'
    if COL_YEARS in df.columns:
        df['장기난임_플래그']       = (df[COL_YEARS].fillna(0) >= 3).astype(int)
        df['나이_시도연수_상호작용'] = age * df[COL_YEARS].fillna(0)

    # ── 시술 횟수 버킷팅 ──────────────────────────────────────────
    df['총시술_버킷'] = pd.cut(df['총 시술 횟수'].fillna(-1),
                                bins=[-2,-0.5,0.5,1.5,2.5,3.5,100],
                                labels=['결측','0회','1회','2회','3회','4회이상']).astype(str)

    # 범주형 NaN → missing
    for col in df.select_dtypes('object').columns:
        df[col] = df[col].fillna('missing')

print('피처 엔지니어링 완료')
print(f'Train: {train.shape} / Test: {test.shape}')

---
# Section 3. 피처 정제 + 교호작용 추가

- 중요도 0 피처 제거
- 3모델 중요도 공통 상위 변수 기반 교호작용 추가

In [ ]:
# 중요도 0 피처 제거
ZERO_IMP_COLS = [
    '배아생성_제로_플래그','배아생성_실패_플래그','난자수집_실패_플래그',
    '난자저장용_포함','배반포이식_여부','불임 원인 - 여성 요인',
    'IVF_배반포_조합','불임 원인 - 정자 면역학적 요인','high_resp_freezeall',
]
for df in [train, test]:
    drop = [c for c in ZERO_IMP_COLS if c in df.columns]
    df.drop(columns=drop, inplace=True)

# 교호작용 추가
for df in [train, test]:
    emb_n   = df['이식된 배아 수'].fillna(0)
    emb_log = df['이식된 배아 수_log'].fillna(0)
    age     = df['나이_순서'].fillna(0)
    quality = df['배아품질_종합점수'].fillna(0)
    is_ivf  = (df['시술 유형'] == 'IVF').astype(int)
    is_tx   = df['이식배아_있음'].fillna(0)
    tx_day  = df['배아 이식 경과일'].replace(-1, np.nan).fillna(0)
    culture = df['배양_기간_일'].replace(-1, np.nan).fillna(0)
    stored  = df['저장된 배아 수'].fillna(0)

    df['이식배아수_나이_교호']      = emb_n * age
    df['이식배아log_나이_교호']     = emb_log * age
    df['이식배아수_품질_교호']      = emb_n * quality
    df['이식배아log_품질_교호']     = emb_log * quality
    df['이식배아수_IVF_교호']       = emb_n * is_ivf
    df['이식성공_품질_교호']        = is_tx * quality
    df['고령자가난자_품질_교호']    = df['자가난자_고령_조합'].fillna(0) * quality
    df['고위험3중_이식배아_교호']   = df['고위험_3중_조합'].fillna(0) * emb_n
    df['이식경과일_이식배아_교호']  = tx_day * emb_n
    df['이식배아수_출산경험_교호']  = emb_n * df['출산경험_있음'].fillna(0)
    df['나이_저장비율_교호']        = age * df['저장_비율'].fillna(0)
    df['잔여배아_품질_교호']        = df['잔여배아_수'].fillna(0) * quality
    df['이식배아수_제곱']           = emb_n ** 2
    df['이식배아_총배아_곱']        = emb_n * df['총 생성 배아 수'].fillna(0)
    df['단일배아이식_여부']         = (emb_n == 1).astype(int)
    df['2개배아이식_여부']          = (emb_n == 2).astype(int)
    df['이식여부_이식경과일_교호']  = is_tx * tx_day
    df['이식여부_배양기간_교호']    = is_tx * culture
    df['이식배아수_이식경과일_교호'] = emb_n * tx_day
    df['이식배아수_배양기간_교호']  = emb_n * culture
    df['이식배아수_저장배아_교호']  = emb_n * stored
    # 시술 시기 코드 교호작용 제외 (랜덤 코드 확인 → 과적합)

print('피처 정제 + 교호작용 완료')
print(f'Train: {train.shape} / Test: {test.shape}')

---
# Section 4. 피처 준비 (3모델용)

In [ ]:
feat_cols = [c for c in train.columns if c not in [TARGET, ID_COL]]

X      = train[feat_cols].copy()
y      = train[TARGET].copy()
X_test = test[feat_cols].copy()

# 범주형 컬럼 탐지 (StringDtype 포함)
cat_features = [
    c for c in feat_cols
    if X[c].dtype == 'object' or pd.api.types.is_string_dtype(X[c])
]
for col in cat_features:
    X[col]      = X[col].astype(str).fillna('missing')
    X_test[col] = X_test[col].astype(str).fillna('missing')

# CatBoost용: 인덱스 번호
cat_feature_indices = [X.columns.get_loc(c) for c in cat_features]

# LightGBM용: category dtype
X_lgb      = X.copy()
X_test_lgb = X_test.copy()
for col in cat_features:
    X_lgb[col]      = X_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')

# XGBoost용: OrdinalEncoder + 중앙값 대체
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_xgb      = X.copy()
X_test_xgb = X_test.copy()
X_xgb[cat_features]      = enc.fit_transform(X[cat_features])
X_test_xgb[cat_features] = enc.transform(X_test[cat_features])
num_features = [c for c in feat_cols if c not in cat_features]
medians = X_xgb[num_features].median()  # train에서만 fit
X_xgb[num_features]      = X_xgb[num_features].fillna(medians)
X_test_xgb[num_features] = X_test_xgb[num_features].fillna(medians)

neg = (y == 0).sum()
pos = (y == 1).sum()
spw = neg / pos

N_SPLITS = 5
print(f'피처: {len(feat_cols)}개 / 범주형: {len(cat_features)}개')
print(f'scale_pos_weight: {spw:.2f}')

---
# Section 5. Seed Averaging — 3모델 5-fold 앙상블

## 파라미터
- CatBoost: Optuna 최적값 (50 trials)
- LightGBM: Optuna 최적값 (50 trials)
- XGBoost: Optuna 최적값 (50 trials)

## Seed Averaging
- 동일 모델을 5개 SEED로 반복 학습 → 분산 감소
- 파라미터 변경 없음 → 과적합 리스크 없음

In [ ]:
SEEDS = [42, 0, 123, 2024, 777]

oof_cat_all  = np.zeros(len(X))
oof_lgb_all  = np.zeros(len(X))
oof_xgb_all  = np.zeros(len(X))
test_cat_all = np.zeros(len(X_test))
test_lgb_all = np.zeros(len(X_test))
test_xgb_all = np.zeros(len(X_test))

seed_results = []

for seed_idx, SEED in enumerate(SEEDS):
    print('\n' + '='*55)
    print(f'SEED {SEED}  ({seed_idx+1}/{len(SEEDS)})')
    print('='*55)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    # ── 파라미터 (Optuna 최적값) ──────────────────────────────────
    cat_params = {
        'iterations': 794, 'learning_rate': 0.030094391673729362,
        'depth': 7, 'l2_leaf_reg': 7.433949857398995,
        'bagging_temperature': 0.3980358270898108,
        'random_strength': 1.3075975210328405, 'border_count': 108,
        'loss_function': 'Logloss', 'eval_metric': 'AUC',
        'scale_pos_weight': spw, 'random_seed': SEED,
        'early_stopping_rounds': 50, 'verbose': False, 'use_best_model': True,
    }
    lgb_params = {
        'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
        'num_leaves': 22, 'max_depth': 7, 'learning_rate': 0.02030926523583015,
        'n_estimators': 956, 'subsample': 0.9916893928795039,
        'colsample_bytree': 0.8760988294276582,
        'reg_alpha': 0.5158539052029842, 'reg_lambda': 0.05557192411355649,
        'min_child_samples': 98,
        'scale_pos_weight': spw, 'random_state': SEED, 'verbose': -1,
    }
    xgb_params = {
        'objective': 'binary:logistic', 'eval_metric': 'auc', 'tree_method': 'hist',
        'max_depth': 4, 'learning_rate': 0.028817004923326044,
        'n_estimators': 835, 'subsample': 0.8755018207309047,
        'colsample_bytree': 0.8959610275285067,
        'reg_alpha': 0.9163081802804784, 'reg_lambda': 1.1472566862699578,
        'min_child_weight': 8, 'gamma': 0.728514692891076,
        'scale_pos_weight': spw, 'random_state': SEED,
        'verbosity': 0, 'early_stopping_rounds': 50,
    }

    oof_cat  = np.zeros(len(X))
    oof_lgb  = np.zeros(len(X))
    oof_xgb  = np.zeros(len(X))
    test_cat = np.zeros(len(X_test))
    test_lgb = np.zeros(len(X_test))
    test_xgb = np.zeros(len(X_test))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr  = X.iloc[tr_idx].reset_index(drop=True)
        X_val = X.iloc[val_idx].reset_index(drop=True)
        y_tr  = y.iloc[tr_idx].reset_index(drop=True)
        y_val = y.iloc[val_idx].reset_index(drop=True)

        # CatBoost
        cb = CatBoostClassifier(**cat_params)
        cb.fit(Pool(X_tr, y_tr, cat_features=cat_feature_indices),
               eval_set=Pool(X_val, y_val, cat_features=cat_feature_indices))
        oof_cat[val_idx]  = cb.predict_proba(Pool(X_val, cat_features=cat_feature_indices))[:, 1]
        test_cat         += cb.predict_proba(Pool(X_test, cat_features=cat_feature_indices))[:, 1] / N_SPLITS

        # LightGBM
        lb = lgb.LGBMClassifier(**lgb_params)
        lb.fit(X_lgb.iloc[tr_idx], y_tr,
               eval_set=[(X_lgb.iloc[val_idx], y_val)],
               callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        oof_lgb[val_idx]  = lb.predict_proba(X_lgb.iloc[val_idx])[:, 1]
        test_lgb         += lb.predict_proba(X_test_lgb)[:, 1] / N_SPLITS

        # XGBoost
        xb = xgb.XGBClassifier(**xgb_params)
        xb.fit(X_xgb.iloc[tr_idx], y_tr,
               eval_set=[(X_xgb.iloc[val_idx], y_val)],
               verbose=False)
        oof_xgb[val_idx]  = xb.predict_proba(X_xgb.iloc[val_idx])[:, 1]
        test_xgb         += xb.predict_proba(X_test_xgb)[:, 1] / N_SPLITS

        print(f'  Fold {fold+1} | CAT: {roc_auc_score(y_val, oof_cat[val_idx]):.5f} '
              f'| LGB: {roc_auc_score(y_val, oof_lgb[val_idx]):.5f} '
              f'| XGB: {roc_auc_score(y_val, oof_xgb[val_idx]):.5f}')

    # 이 SEED의 최적 가중치
    best_auc, best_w = 0.0, (0.6, 0.15, 0.25)
    for w1 in np.arange(0.0, 1.05, 0.05):
        for w2 in np.arange(0.0, 1.05-w1, 0.05):
            w3 = round(1-w1-w2, 2)
            if w3 < 0: continue
            auc = roc_auc_score(y, oof_cat*w1 + oof_lgb*w2 + oof_xgb*w3)
            if auc > best_auc:
                best_auc, best_w = auc, (w1, w2, w3)

    print(f'SEED {SEED} 앙상블 AUC: {best_auc:.5f} '
          f'(w_cat={best_w[0]:.2f}, w_lgb={best_w[1]:.2f}, w_xgb={best_w[2]:.2f})')
    seed_results.append({'seed': SEED, 'auc': best_auc, 'w': best_w})

    oof_cat_all  += oof_cat  / len(SEEDS)
    oof_lgb_all  += oof_lgb  / len(SEEDS)
    oof_xgb_all  += oof_xgb  / len(SEEDS)
    test_cat_all += test_cat / len(SEEDS)
    test_lgb_all += test_lgb / len(SEEDS)
    test_xgb_all += test_xgb / len(SEEDS)

print('\n모든 SEED 학습 완료')

---
# Section 6. 최종 앙상블 + 제출 파일 생성

In [ ]:
# SEED별 결과 출력
print('=== SEED별 앙상블 AUC ===')
for r in seed_results:
    print(f"  SEED {r['seed']:4d} | AUC: {r['auc']:.5f}")
print(f"  평균: {np.mean([r['auc'] for r in seed_results]):.5f}")
print(f"  편차: {np.std([r['auc'] for r in seed_results]):.5f}")

# 평균된 OOF로 최적 가중치 재탐색
best_auc_avg, best_w_avg = 0.0, (0.6, 0.15, 0.25)
for w1 in np.arange(0.0, 1.05, 0.05):
    for w2 in np.arange(0.0, 1.05-w1, 0.05):
        w3 = round(1-w1-w2, 2)
        if w3 < 0: continue
        blended = oof_cat_all*w1 + oof_lgb_all*w2 + oof_xgb_all*w3
        auc = roc_auc_score(y, blended)
        if auc > best_auc_avg:
            best_auc_avg = auc
            best_w_avg = (w1, w2, w3)

print(f'\n✅ Seed Averaging OOF AUC: {best_auc_avg:.5f}')
print(f'   최적 가중치: CAT={best_w_avg[0]:.2f}, LGB={best_w_avg[1]:.2f}, XGB={best_w_avg[2]:.2f}')

# 제출 파일
test_blended = (test_cat_all * best_w_avg[0] +
                test_lgb_all * best_w_avg[1] +
                test_xgb_all * best_w_avg[2])

sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
submission = sample_sub.copy()
submission[sample_sub.columns[1]] = test_blended
submission.to_csv('submission_final.csv', index=False, encoding='utf-8-sig')

print('\n저장 완료: submission_final.csv')
display(submission.head())